### Proyecto 3 - Implementación de un DATA LAKE sobre HDFS y procesamiento de datos con HIVE

#### Diagrama ETL

[![Screenshot-6.png](https://i.postimg.cc/pV05D6zc/Screenshot-6.png)](https://postimg.cc/DmXzh5C1)

#### Paso 1 - Creación de estructura de directorios

In [ ]:
-- Crear la estructura de directorios sobre HDFS
hdfs dfs -mkdir -p \
/user/proyectos/proyecto3/database/landing_tmp/persona \
/user/proyectos/proyecto3/database/landing_tmp/empresa \
/user/proyectos/proyecto3/database/landing_tmp/transaccion 

hdfs dfs -mkdir -p \
/user/proyectos/proyecto3/database/landing/persona \
/user/proyectos/proyecto3/database/landing/empresa \
/user/proyectos/proyecto3/database/landing/transaccion

hdfs dfs -mkdir -p \
/user/proyectos/proyecto3/database/universal/persona \
/user/proyectos/proyecto3/database/universal/empresa \
/user/proyectos/proyecto3/database/universal/transaccion \
/user/proyectos/proyecto3/database/universal/transaccion_enriquecida

hdfs dfs -mkdir -p \
/user/proyectos/proyecto3/database/smart/transaccion_por_edad \
/user/proyectos/proyecto3/database/smart/transaccion_por_trabajo \
/user/proyectos/proyecto3/database/smart/transaccion_por_empresa 

-- Subimos los archivos de esquemas que se utilizaran en la creacion de la capa LANDING
hdfs dfs -mkdir -p /user/proyectos/proyecto3/schema/landing

hdfs dfs -put \
/home/hadoop/data/persona.avsc \
/home/hadoop/data/empresa.avsc \
/home/hadoop/data/transaccion.avsc \
/user/proyectos/proyecto3/schema/landing

#### Paso 2 - Crear bases de datos

In [ ]:
-- En la consola de HIVE, adaptar y ejecutar:
CREATE DATABASE IF NOT EXISTS LANDING_TMP LOCATION '/user/proyectos/proyecto3/database/landing_tmp';
CREATE DATABASE IF NOT EXISTS LANDING LOCATION '/user/proyectos/proyecto3/database/landing';
CREATE DATABASE IF NOT EXISTS UNIVERSAL LOCATION '/user/proyectos/proyecto3/database/universal';
CREATE DATABASE IF NOT EXISTS SMART LOCATION '/user/proyectos/proyecto3/database/smart';

#### Paso 3 - Crear la capa LANDING_TMP

In [ ]:
                                              ------------
                                                 PERSONA    
                                              ------------
-- En la consola de HIVE, adaptar y ejecutar
CREATE TABLE LANDING_TMP.PERSONA(
ID STRING,
NOMBRE STRING,
TELEFONO STRING,
CORREO STRING,
FECHA_INGRESO STRING,
EDAD STRING,
SALARIO STRING,
ID_EMPRESA STRING
)
ROW FORMAT DELIMITED
FIELDS TERMINATED BY '|'
LINES TERMINATED BY '\n'
STORED AS TEXTFILE
LOCATION '/user/proyectos/proyecto3/database/landing_tmp/persona';

-- Subida de datos
-- En la consola HADOOP, adaptar y ejecutar
hdfs dfs -put /home/hadoop/data/persona.data /user/proyectos/proyecto3/database/landing_tmp/persona

-- O desde HIVE utilizando (es lo mismo)
LOAD DATA LOCAL INPATH '/home/hadoop/data/persona.data' INTO TABLE LANDING_TMP.PERSONA;

-- En la consola HADOOP o en HIVE, verificar que existan los datos en la tabla
SELECT * FROM LANDING_TMP.PERSONA LIMIT 10;

+-------------+-----------------+-------------------+-----------------------------------------+------------------------+---------------+------------------+---------------------+
| persona.id  | persona.nombre  | persona.telefono  |             persona.correo              | persona.fecha_ingreso  | persona.edad  | persona.salario  | persona.id_empresa  |
+-------------+-----------------+-------------------+-----------------------------------------+------------------------+---------------+------------------+---------------------+
| ID          | NOMBRE          | TELEFONO          | CORREO                                  | FECHA_INGRESO          | EDAD          | SALARIO          | ID_EMPRESA          |
| 1           | Carl            | 1-745-633-9145    | arcu.Sed.et@ante.co.uk                  | 2004-04-23             | 32            | 20095            | 5                   |
| 2           | Priscilla       | 155-2498          | Donec.egestas.Aliquam@volutpatnunc.edu  | 2019-02-17             | 34            | 9298             | 2                   |
| 3           | Jocelyn         | 1-204-956-8594    | amet.diam@lobortis.co.uk                | 2002-08-01             | 27            | 10853            | 3                   |
| 4           | Aidan           | 1-719-862-9385    | euismod.et.commodo@nibhlaciniaorci.edu  | 2018-11-06             | 29            | 3387             | 10                  |
| 5           | Leandra         | 839-8044          | at@pretiumetrutrum.com                  | 2002-10-10             | 41            | 22102            | 1                   |
| 6           | Bert            | 797-4453          | a.felis.ullamcorper@arcu.org            | 2017-04-25             | 70            | 7800             | 7                   |
| 7           | Mark            | 1-680-102-6792    | Quisque.ac@placerat.ca                  | 2006-04-21             | 52            | 8112             | 5                   |
| 8           | Jonah           | 214-2975          | eu.ultrices.sit@vitae.ca                | 2017-10-07             | 23            | 17040            | 5                   |
| 9           | Hanae           | 935-2277          | eu@Nunc.ca                              | 2003-05-25             | 69            | 6834             | 3                   |
+-------------+-----------------+-------------------+-----------------------------------------+------------------------+---------------+------------------+---------------------+

-- (Se cargara el "ENCABEZADO" como un registro más, eso no es correcto, sin embargo, este tipo de  errores se CORRIGEN EN LA ETAPA "UNIVERSAL")

-- Repetir los pasos anteriores para las tablas "EMPRESA" y "TRANSACCION" 

                                              ------------
                                                 EMPRESA    
                                              ------------
-- En la consola de HIVE, adaptar y ejecutar
CREATE TABLE LANDING_TMP.EMPRESA(
ID STRING,
NOMBRE STRING
) 
ROW FORMAT DELIMITED
FIELDS TERMINATED BY '|'
LINES TERMINATED BY '\n'
STORED AS TEXTFILE
LOCATION '/user/proyectos/proyecto3/database/landing_tmp/empresa';

-- Subida de datos
-- En la consola HADOOP, adaptar y ejecutar
hdfs dfs -put /home/hadoop/data/empresa.data /user/proyectos/proyecto3/database/landing_tmp/empresa

-- O desde HIVE utilizando (es lo mismo)
LOAD DATA LOCAL INPATH '/home/hadoop/data/empresa.data' INTO TABLE LANDING_TMP.EMPRESA;  

-- En la consola HADOOP o en HIVE, verificar que existan los datos en la tabla
SELECT * FROM LANDING_TMP.EMPRESA LIMIT 10;

+-------------+-----------------+
| empresa.id  | empresa.nombre  |
+-------------+-----------------+
| ID          | NOMBRE          |
| 1           | Walmart         |
| 2           | Microsoft       |
| 3           | Apple           |
| 4           | Toyota          |
| 5           | Amazon          |
| 6           | Google          |
| 7           | Samsung         |
| 8           | HP              |
| 9           | IBM             |
+-------------+-----------------+


                                              -------------
                                               TRANSACCION    
                                              -------------
-- En la consola de HIVE, adaptar y ejecutar
CREATE TABLE LANDING_TMP.TRANSACCION(
ID_PERSONA STRING,
ID_EMPRESA STRING,
MONTO STRING,
FECHA STRING  <------------------------------------- Agrego este campo, siendo que no existe en el archivo de definicion de esquema ¿Por que?
)                                                    Porque al crear una tabla particionada, no se incluye el campo particionado, en este caso,
ROW FORMAT DELIMITED                                 el campo FECHA (En la capa LANDING)
FIELDS TERMINATED BY '|'
LINES TERMINATED BY '\n'
STORED AS TEXTFILE
LOCATION '/user/proyectos/proyecto3/database/landing_tmp/transaccion';

-- Subida de datos
-- En la consola HADOOP, adaptar y ejecutar
hdfs dfs -put /home/hadoop/data/transacciones.data /user/proyectos/proyecto3/database/landing_tmp/transaccion 

-- O desde HIVE utilizando (es lo mismo)
LOAD DATA LOCAL INPATH '/home/hadoop/data/transacciones.data' INTO TABLE LANDING_TMP.TRANSACCION;

-- En la consola HADOOP o en HIVE, verificar que existan los datos en la tabla
SELECT * FROM LANDING_TMP.TRANSACCION LIMIT 10;

+-------------------------+-------------------------+--------------------+--------------------+
| transaccion.id_persona  | transaccion.id_empresa  | transaccion.monto  | transaccion.fecha  |
+-------------------------+-------------------------+--------------------+--------------------+
| ID_PERSONA              | ID_EMPRESA              | MONTO              | FECHA              |
| 18                      | 3                       | 1383               | 2018-01-21         |
| 30                      | 6                       | 2331               | 2018-01-21         |
| 47                      | 2                       | 2280               | 2018-01-21         |
| 28                      | 1                       | 730                | 2018-01-21         |
| 91                      | 4                       | 3081               | 2018-01-21         |
| 74                      | 8                       | 2409               | 2018-01-21         |
| 41                      | 2                       | 3754               | 2018-01-22         |
| 42                      | 9                       | 4079               | 2018-01-22         |
| 24                      | 6                       | 4475               | 2018-01-22         |
+-------------------------+-------------------------+--------------------+--------------------+


#### Paso 4 - Crear la capa LANDING

In [ ]:
                                              ------------
                                                 PERSONA    
                                              ------------
------------------
ACTIVAR COMPRESION
------------------

-- Desde HIVE, activamos la compresión y el formato SNAPPY
SET hive.exec.compress.output=true;
SET avro.output.codec=snappy;

-----------
CREAR TABLA
-----------

-- Desde HIVE, creamos la tabla PERSONA 
CREATE TABLE LANDING.PERSONA
STORED AS AVRO
LOCATION '/user/proyectos/proyecto3/database/landing/persona'
TBLPROPERTIES (
'avro.schema.url'='hdfs:///user/proyectos/proyecto3/schema/landing/persona.avsc',
'avro.output.codec'='snappy'
);

------------------------------
INSERCION DE DATOS EN LA TABLA
------------------------------

-- Desde HIVE, ejecutamos la inserción de datos desde la tabla "LANDING_TMP" a la tabla "LANDING"
INSERT OVERWRITE TABLE LANDING.PERSONA SELECT * FROM LANDING_TMP.PERSONA;

-- Desde HIVE, verificamos que se hayan insertado los datos en la tabla "LANDING"
SELECT * FROM LANDING.PERSONA LIMIT 10;  

+-------------+-----------------+-------------------+-----------------------------------------+------------------------+---------------+------------------+---------------------+
| persona.id  | persona.nombre  | persona.telefono  |             persona.correo              | persona.fecha_ingreso  | persona.edad  | persona.salario  | persona.id_empresa  |
+-------------+-----------------+-------------------+-----------------------------------------+------------------------+---------------+------------------+---------------------+
| ID          | NOMBRE          | TELEFONO          | CORREO                                  | FECHA_INGRESO          | EDAD          | SALARIO          | ID_EMPRESA          |
| 1           | Carl            | 1-745-633-9145    | arcu.Sed.et@ante.co.uk                  | 2004-04-23             | 32            | 20095            | 5                   |
| 2           | Priscilla       | 155-2498          | Donec.egestas.Aliquam@volutpatnunc.edu  | 2019-02-17             | 34            | 9298             | 2                   |
| 3           | Jocelyn         | 1-204-956-8594    | amet.diam@lobortis.co.uk                | 2002-08-01             | 27            | 10853            | 3                   |
| 4           | Aidan           | 1-719-862-9385    | euismod.et.commodo@nibhlaciniaorci.edu  | 2018-11-06             | 29            | 3387             | 10                  |
| 5           | Leandra         | 839-8044          | at@pretiumetrutrum.com                  | 2002-10-10             | 41            | 22102            | 1                   |
| 6           | Bert            | 797-4453          | a.felis.ullamcorper@arcu.org            | 2017-04-25             | 70            | 7800             | 7                   |
| 7           | Mark            | 1-680-102-6792    | Quisque.ac@placerat.ca                  | 2006-04-21             | 52            | 8112             | 5                   |
| 8           | Jonah           | 214-2975          | eu.ultrices.sit@vitae.ca                | 2017-10-07             | 23            | 17040            | 5                   |
| 9           | Hanae           | 935-2277          | eu@Nunc.ca                              | 2003-05-25             | 69            | 6834             | 3                   |
+-------------+-----------------+-------------------+-----------------------------------------+------------------------+---------------+------------------+---------------------+


                                              ------------
                                                 EMPRESA    
                                              ------------
------------------
ACTIVAR COMPRESION
------------------

-- Desde HIVE, activamos la compresión y el formato SNAPPY (Ya lo hicimos, pero para recordar)
SET hive.exec.compress.output=true;
SET avro.output.codec=snappy;

-----------
CREAR TABLA
-----------

-- Desde HIVE, creamos la tabla EMPRESA 
CREATE TABLE LANDING.EMPRESA
STORED AS AVRO
LOCATION '/user/proyectos/proyecto3/database/landing/empresa'
TBLPROPERTIES (
'avro.schema.url'='hdfs:///user/proyectos/proyecto3/schema/landing/empresa.avsc',
'avro.output.codec'='snappy'
);

------------------------------
INSERCION DE DATOS EN LA TABLA
------------------------------

-- Desde HIVE, ejecutamos la inserción de datos desde la tabla "LANDING_TMP" a la tabla "LANDING"
INSERT OVERWRITE TABLE LANDING.EMPRESA SELECT * FROM LANDING_TMP.EMPRESA;

-- Desde HIVE, verificamos que se hayan insertado los datos en la tabla "LANDING"
SELECT * FROM LANDING.EMPRESA LIMIT 10;  

+-------------+-----------------+
| empresa.id  | empresa.nombre  |
+-------------+-----------------+
| ID          | NOMBRE          |
| 1           | Walmart         |
| 2           | Microsoft       |
| 3           | Apple           |
| 4           | Toyota          |
| 5           | Amazon          |
| 6           | Google          |
| 7           | Samsung         |
| 8           | HP              |
| 9           | IBM             |
+-------------+-----------------+


                                            ----------------
                                               TRANSACCION    
                                            ----------------
------------------
ACTIVAR COMPRESION
------------------

-- Desde HIVE, activamos la compresión y el formato SNAPPY (Ya lo hicimos, pero para recordar)
SET hive.exec.compress.output=true;
SET avro.output.codec=snappy;

---------------------------------
ACTIVAR PARTICIONAMIENTO DINAMICO
---------------------------------

SET hive.exec.dynamic.partition=true;
SET hive.exec.dynamic.partition.mode=nonstrict;

-----------
CREAR TABLA
-----------

Queremos crear una tabla particionada. Las particiones NO SON FLEXIBLES aunque esten en un AVRO, son
ESTATICAS. Porque a nivel de HDFS, una particion es un Subdirectorio de HDFS, asi que es algo estatico.

-- Desde HIVE, creamos la tabla TRANSACCION 
CREATE TABLE LANDING.TRANSACCION
PARTITIONED BY (FECHA STRING)
STORED AS AVRO
LOCATION '/user/proyectos/proyecto3/database/landing/transaccion'
TBLPROPERTIES (
'avro.schema.url'='hdfs:///user/proyectos/proyecto3/schema/landing/transaccion.avsc',
'avro.output.codec'='snappy'
);

------------------------------
INSERCION DE DATOS EN LA TABLA
------------------------------

-- Desde HIVE, ejecutamos la inserción de datos desde la tabla "LANDING_TMP" a la tabla "LANDING"
INSERT OVERWRITE TABLE LANDING.TRANSACCION
PARTITION(FECHA)
SELECT * FROM  LANDING_TMP.TRANSACCION;

-- Esta vez, como el particionado dinámico es true, Hive comprobará los valores de la columna de particionamiento y creará 
-- una partición separada para cada valor único de esa columna:

                        /user/proyectos/proyecto3/database/landing/transaccion/fecha=2018-01-21
                        /user/proyectos/proyecto3/database/landing/transaccion/fecha=2018-01-22
                        /user/proyectos/proyecto3/database/landing/transaccion/fecha=2018-01-23
                        /user/proyectos/proyecto3/database/landing/transaccion/fecha=FECHA

-- Desde HIVE, verificamos que se hayan insertado los datos en la tabla "LANDING"
SELECT * FROM LANDING.TRANSACCION LIMIT 10;

+-------------------------+-------------------------+--------------------+--------------------+
| transaccion.id_persona  | transaccion.id_empresa  | transaccion.monto  | transaccion.fecha  |
+-------------------------+-------------------------+--------------------+--------------------+
| 18                      | 3                       | 1383               | 2018-01-21         |
| 30                      | 6                       | 2331               | 2018-01-21         |
| 47                      | 2                       | 2280               | 2018-01-21         |
| 28                      | 1                       | 730                | 2018-01-21         |
| 91                      | 4                       | 3081               | 2018-01-21         |
| 74                      | 8                       | 2409               | 2018-01-21         |
| 83                      | 5                       | 2079               | 2018-01-21         |
| 48                      | 4                       | 2543               | 2018-01-21         |
| 15                      | 6                       | 1434               | 2018-01-21         |
| 89                      | 4                       | 780                | 2018-01-21         |
+-------------------------+-------------------------+--------------------+--------------------+


-- Verificamos
SHOW PARTITIONS LANDING.TRANSACCION;
--   ________________
--  |   partition    |
--  |----------------|
--  |fecha=2018-01-21|
--  |fecha=2018-01-22|
--  |fecha=2018-01-23|
--  |fecha=FECHA     | <---------------- Debemos borrar esta particion erronea, eso lo
--  |________________|                   haremos en la capa UNIVERSAL.


#### Paso 5 - Crear la capa UNIVERSAL

In [ ]:
                                              ------------
                                                 PERSONA    
                                              ------------
------------------
ACTIVAR COMPRESION
------------------

-- Desde HIVE, activamos la compresión y el formato SNAPPY
SET hive.exec.compress.output=true;
SET parquet.compression=SNAPPY;

-----------
CREAR TABLA
-----------

-- Desde HIVE, creamos la tabla PERSONA (Ahora utilizamos los tipos de datos correctos)
CREATE TABLE UNIVERSAL.PERSONA(
ID STRING,
NOMBRE STRING,
TELEFONO STRING,
CORREO STRING,
FECHA_INGRESO STRING,
EDAD INT,
SALARIO DOUBLE,
ID_EMPRESA STRING
)
STORED AS PARQUET
LOCATION '/user/proyectos/proyecto3/database/universal/persona'
TBLPROPERTIES ("parquet.compression"="SNAPPY");

------------------------------
INSERCION DE DATOS EN LA TABLA
------------------------------

-- Desde HIVE, ejecutamos la inserción de datos desde la tabla "LANDING" a la tabla "UNIVERSAL"
INSERT OVERWRITE TABLE UNIVERSAL.PERSONA
SELECT
  cast(ID AS STRING),
  cast(NOMBRE AS STRING),
  cast(TELEFONO AS STRING),
  cast(CORREO AS STRING),
  cast(FECHA_INGRESO AS STRING),
  cast(EDAD AS INT),
  cast(SALARIO AS DOUBLE),
  cast(ID_EMPRESA AS STRING)
FROM 
  LANDING.PERSONA
WHERE 
  ID != 'ID';    <-------------------------- Con esta linea filtramos ese registro que nos mostraba
                                             los encabezados como un registro más.
 EDAD > 0        <-------------------------- Es aqui donde podemos comenzar a restringir los datos
 SALARIO > 1000                              (Estas restricciones son solo para ejemplo, no tomar en cuenta)

-- Desde HIVE, verificamos que se hayan insertado los datos en la tabla "LANDING"
SELECT * FROM UNIVERSAL.PERSONA LIMIT 10;

+-------------+-----------------+-------------------+-----------------------------------------+------------------------+---------------+------------------+---------------------+
| persona.id  | persona.nombre  | persona.telefono  |             persona.correo              | persona.fecha_ingreso  | persona.edad  | persona.salario  | persona.id_empresa  |
+-------------+-----------------+-------------------+-----------------------------------------+------------------------+---------------+------------------+---------------------+
| 1           | Carl            | 1-745-633-9145    | arcu.Sed.et@ante.co.uk                  | 2004-04-23             | 32            | 20095.0          | 5                   |
| 2           | Priscilla       | 155-2498          | Donec.egestas.Aliquam@volutpatnunc.edu  | 2019-02-17             | 34            | 9298.0           | 2                   |
| 3           | Jocelyn         | 1-204-956-8594    | amet.diam@lobortis.co.uk                | 2002-08-01             | 27            | 10853.0          | 3                   |
| 4           | Aidan           | 1-719-862-9385    | euismod.et.commodo@nibhlaciniaorci.edu  | 2018-11-06             | 29            | 3387.0           | 10                  |
| 5           | Leandra         | 839-8044          | at@pretiumetrutrum.com                  | 2002-10-10             | 41            | 22102.0          | 1                   |
| 6           | Bert            | 797-4453          | a.felis.ullamcorper@arcu.org            | 2017-04-25             | 70            | 7800.0           | 7                   |
| 7           | Mark            | 1-680-102-6792    | Quisque.ac@placerat.ca                  | 2006-04-21             | 52            | 8112.0           | 5                   |
| 8           | Jonah           | 214-2975          | eu.ultrices.sit@vitae.ca                | 2017-10-07             | 23            | 17040.0          | 5                   |
| 9           | Hanae           | 935-2277          | eu@Nunc.ca                              | 2003-05-25             | 69            | 6834.0           | 3                   |
| 10          | Cadman          | 1-866-561-2701    | orci.adipiscing.non@semperNam.ca        | 2001-05-19             | 19            | 7996.0           | 7                   |
+-------------+-----------------+-------------------+-----------------------------------------+------------------------+---------------+------------------+---------------------+

-- Verificamos los tipos de datos de sus campos
DESCRIBE UNIVERSAL.PERSONA;

+----------------+------------+----------+
|    col_name    | data_type  | comment  |
+----------------+------------+----------+
| id             | string     |          |
| nombre         | string     |          |
| telefono       | string     |          |
| correo         | string     |          |
| fecha_ingreso  | string     |          |
| edad           | int        |          |
| salario        | double     |          |
| id_empresa     | string     |          |
+----------------+------------+----------+


                                              ------------
                                                 EMPRESA    
                                              ------------
------------------
ACTIVAR COMPRESION
------------------

-- Desde HIVE, activamos la compresión y el formato SNAPPY
SET hive.exec.compress.output=true;
SET parquet.compression=SNAPPY;

-----------
CREAR TABLA
-----------

-- Desde HIVE, creamos la tabla EMPRESA (Ahora utilizamos los tipos de datos correctos)
CREATE TABLE UNIVERSAL.EMPRESA(
ID STRING,
NOMBRE STRING
)
STORED AS PARQUET
LOCATION '/user/proyectos/proyecto3/database/universal/empresa'
TBLPROPERTIES ("parquet.compression"="SNAPPY");

------------------------------
INSERCION DE DATOS EN LA TABLA
------------------------------

-- Desde HIVE, ejecutamos la inserción de datos desde la tabla "LANDING" a la tabla "UNIVERSAL"
INSERT OVERWRITE TABLE UNIVERSAL.EMPRESA
SELECT
  cast(ID AS STRING),
  cast(NOMBRE AS STRING)
FROM 
  LANDING.EMPRESA
WHERE 
  ID != 'ID';    <-------------------------- Con esta linea filtramos ese registro que nos mostraba
                                             los encabezados como un registro más.

-- Desde HIVE, verificamos que se hayan insertado los datos en la tabla "LANDING"
SELECT * FROM UNIVERSAL.EMPRESA LIMIT 10;

+-------------+-----------------+
| empresa.id  | empresa.nombre  |
+-------------+-----------------+
| 1           | Walmart         |
| 2           | Microsoft       |
| 3           | Apple           |
| 4           | Toyota          |
| 5           | Amazon          |
| 6           | Google          |
| 7           | Samsung         |
| 8           | HP              |
| 9           | IBM             |
| 10          | Sony            |
+-------------+-----------------+

-- Verificamos los tipos de datos de sus campos
DESCRIBE UNIVERSAL.EMPRESA;

+-----------+------------+----------+
| col_name  | data_type  | comment  |
+-----------+------------+----------+
| id        | string     |          |
| nombre    | string     |          |
+-----------+------------+----------+


                                            ----------------
                                               TRANSACCION    
                                            ----------------
------------------
ACTIVAR COMPRESION 
------------------

-- Desde HIVE, activamos la compresión y el formato SNAPPY
SET hive.exec.compress.output=true;
SET parquet.compression=SNAPPY;

---------------------------------
ACTIVAR PARTICIONAMIENTO DINAMICO
---------------------------------

SET hive.exec.dynamic.partition=true;
SET hive.exec.dynamic.partition.mode=nonstrict;

-----------
CREAR TABLA
-----------

-- Desde HIVE, creamos la tabla TRANSACCION (Ahora utilizamos los tipos de datos correctos)
CREATE TABLE UNIVERSAL.TRANSACCION(
ID_PERSONA STRING,
ID_EMPRESA STRING,
MONTO DOUBLE
)
PARTITIONED BY (FECHA STRING)
STORED AS PARQUET
LOCATION '/user/proyectos/proyecto3/database/universal/transaccion'
TBLPROPERTIES ("parquet.compression"="SNAPPY");

------------------------------
INSERCION DE DATOS EN LA TABLA
------------------------------

-- Desde HIVE, ejecutamos la inserción de datos desde la tabla "LANDING" a la tabla "UNIVERSAL"
INSERT OVERWRITE TABLE UNIVERSAL.TRANSACCION
PARTITION(FECHA)
SELECT
  cast(ID_PERSONA AS STRING),
  cast(ID_EMPRESA AS STRING),
  cast(MONTO AS DOUBLE),
  cast(FECHA AS STRING)
FROM 
  LANDING.TRANSACCION
WHERE 
  ID_PERSONA != 'ID_PERSONA';    <-------------------------- Con esta linea filtramos ese registro que nos mostraba
                                                             los encabezados como un registro más.

-- Esta vez, como el particionado dinámico es true, Hive comprobará los valores de la columna de particionamiento y creará 
-- una partición separada para cada valor único de esa columna:

                        /user/proyectos/proyecto3/database/universal/transaccion/fecha=2018-01-21
                        /user/proyectos/proyecto3/database/universal/transaccion/fecha=2018-01-22
                        /user/proyectos/proyecto3/database/universal/transaccion/fecha=2018-01-23

-- Desde HIVE, verificamos que se hayan insertado los datos en la tabla "LANDING"
SELECT * FROM UNIVERSAL.TRANSACCION LIMIT 10;

+-------------------------+-------------------------+--------------------+--------------------+
| transaccion.id_persona  | transaccion.id_empresa  | transaccion.monto  | transaccion.fecha  |
+-------------------------+-------------------------+--------------------+--------------------+
| 18                      | 3                       | 1383.0             | 2018-01-21         |
| 30                      | 6                       | 2331.0             | 2018-01-21         |
| 47                      | 2                       | 2280.0             | 2018-01-21         |
| 28                      | 1                       | 730.0              | 2018-01-21         |
| 91                      | 4                       | 3081.0             | 2018-01-21         |
| 74                      | 8                       | 2409.0             | 2018-01-21         |
| 83                      | 5                       | 2079.0             | 2018-01-21         |
| 48                      | 4                       | 2543.0             | 2018-01-21         |
| 15                      | 6                       | 1434.0             | 2018-01-21         |
| 89                      | 4                       | 780.0              | 2018-01-21         |
+-------------------------+-------------------------+--------------------+--------------------+

-- Verificamos
SHOW PARTITIONS UNIVERSAL.TRANSACCION;
--   ________________
--  |   partition    |
--  |----------------|
--  |fecha=2018-01-21|
--  |fecha=2018-01-22|
--  |fecha=2018-01-23|
--  |________________|

-- Verificamos los tipos de datos de sus campos
DESCRIBE UNIVERSAL.TRANSACCION;

+--------------------------+------------+----------+
|         col_name         | data_type  | comment  |
+--------------------------+------------+----------+
| id_persona               | string     |          |
| id_empresa               | string     |          |
| monto                    | double     |          |
| fecha                    | string     |          |
|                          | NULL       | NULL     |
| # Partition Information  | NULL       | NULL     |
| # col_name               | data_type  | comment  |
| fecha                    | string     |          |
+--------------------------+------------+----------+


                                              ---------------------------
                                                TRANSACCION_ENRIQUECIDA
                                              ---------------------------
------------------
ACTIVAR COMPRESION 
------------------

-- Desde HIVE, activamos la compresión y el formato SNAPPY
SET hive.exec.compress.output=true;
SET parquet.compression=SNAPPY;

---------------------------------
ACTIVAR PARTICIONAMIENTO DINAMICO
---------------------------------

SET hive.exec.dynamic.partition=true;
SET hive.exec.dynamic.partition.mode=nonstrict;

-----------
CREAR TABLA
-----------

-- Desde HIVE, creamos la tabla TRANSACCION_ENRIQUECIDA
CREATE TABLE UNIVERSAL.TRANSACCION_ENRIQUECIDA(
ID_PERSONA INT,
NOMBRE_PERSONA STRING,
EDAD_PERSONA INT,
SALARIO_PERSONA DOUBLE,
TRABAJO_PERSONA STRING,
MONTO_TRANSACCION DOUBLE,
EMPRESA_TRANSACCION STRING
)
PARTITIONED BY (FECHA_TRANSACCION STRING)
STORED AS PARQUET
LOCATION '/user/proyectos/proyecto3/database/universal/transaccion_enriquecida'
TBLPROPERTIES ("parquet.compression"="SNAPPY");


#### Paso 6 - Crear una tabla desnormalizada sobre la capa UNIVERSAL

In [ ]:
En la capa "universal" crear la tabla "transaccion_enriquecida" la cual debe ser construida en función de las 
tablas "persona", "empresa" y "transaccion". Los campos de la tabla son los siguientes:

               _____________________________________________________________________________________
              |        CAMPO        |  TIPO  |                    DESCRIPCION                       |
              |---------------------|--------|------------------------------------------------------|
              | ID_PERSONA          | INT    | ID de la persona que realizo la transaccion          |
              |---------------------|--------|------------------------------------------------------|
              | NOMBRE_PERSONA      | STRING | Nombre de la persona que realizó la transaccion      |
              |---------------------|--------|------------------------------------------------------|
              | EDAD_PERSONA        | INT    | Edad de la persona que realizó la transaccion        |
              |---------------------|--------|------------------------------------------------------|
              | SALARIO_PERSONA     | DOUBLE | Salario de la persona que realizó la transaccion     |
              |---------------------|--------|------------------------------------------------------|
              | TRABAJO_PERSONA     | STRING | Nombre de la empresa donde trabaja la persona        |
              |---------------------|--------|------------------------------------------------------|
              | MONTO_TRANSACCION   | DOUBLE | Monto de la transaccion realizada                    |
              |---------------------|--------|------------------------------------------------------|
              | FECHA_TRANSACCION   | STRING | Fecha de la transaccion realizada                    |
              |---------------------|--------|------------------------------------------------------|
              | EMPRESA_TRANSACCION | STRING | Nombre de la empresa donde se realizó la transaccion |
              |_____________________|________|______________________________________________________|


----------------------------------------------
DIAGRAMA INPUT/INTERMEDIATE/OUTPUT [PROCESO 1]
----------------------------------------------

TRANSACCION_PERSONA_ENRIQUECIDA_1         -----.
TRANSACCION_PERSONA_ENRIQUECIDA_2              |------> TABLAS TEMPORALES 
TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA   -----'    Seran creadas como tablas temporales
                                                    (al cerrar sesion se borran automaticamente)
                                                    porque solo nos sirven para obtener la     
                                                    tabla output "TRANSACCION_ENRIQUECIDA"

                                                                                                    CAPA UNIVERSAL                                                    
                                                                                                    ==============

TRANSACCION                                 TRANSACCION_PERSONA_ENRIQUECIDA_1
 ________                                              ________    ID_PERSONA
|__|__|__|                                            |__|__|__|   NOMBRE_PERSONA
|__|__|__|--------.                        .--------> |__|__|__|   EDAD_PERSONA
|__|__|__|        |                        |          |__|__|__|   SALARIO_PERSONA
                  |         ______         |               |       ID_EMPRESA_PERSONA ----> Hace referencia al ID de la Empresa donde la Persona trabaja 
PERSONA           |--------| JOIN |--------'               |       MONTO_TRANSACCION
 ________         |        |______|                        |       FECHA_TRANSACCION 
|__|__|__|        |   TRANSACCION.ID_PERSONA               |       ID_EMPRESA_TRANSACCION ----> Hace referencia al ID de la Empresa donde se realizo la transaccion
|__|__|__|--------'             =                          | 
|__|__|__|                  PERSONA.ID                     | 
                                                           |                                                                                                                                                TRANSACCION_ENRIQUECIDA                                                  
EMPRESA                                                    |                         TRANSACCION_PERSONA_ENRIQUECIDA_2                                                                                             ________ 
 ________                                                __˅___                                  ________  ID_PERSONA                                                                       ________              |__|__|__|  
|__|__|__|                                              | JOIN |                                |__|__|__| NOMBRE_PERSONA                                    .---------------------------> | INSERT |-----------> |__|__|__|
|__|__|__|--------------------------------------------->|______|------------------------------> |__|__|__| EDAD_PERSONA                                      |                             |________|             |__|__|__|
|__|__|__|                             TRANSACCION_PERSONA_ENRIQUECIDA_1.ID_EMPRESA             |__|__|__| SALARIO_PERSONA                                   | 
    |                                                       =                                       |      TRABAJO_PERSONA                                   |   
    |                                                   EMPRESA.ID                                  |      MONTO_TRANSACCION                                 | 
    |                                                                                               |      FECHA_TRANSACCION                                 |
    |                                                                                               |      EMPRESA_TRANSACCION       TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA
    |                                                                                               |                                                    ____|___  ID_PERSONA
    |                                                                                            ___˅__                                                 |__|__|__| NOMBRE_PERSONA
--  '-----------------------------------------------------------------------------------------> | JOIN |----------------------------------------------> |__|__|__| EDAD_PERSONA
                                                                                                |______|                                                |__|__|__| SALARIO_PERSONA
                                                                      TRANSACCION_PERSONA_ENRIQUECIDA_2.ID_EMPRESA_TRANSACCION                                     TRABAJO_PERSONA 
                                                                                                   =                                                               MONTO_TRANSACCION 
                                                                                               EMPRESA.ID                                                          FECHA_TRANSACCION 
                                                                                                                                                                   EMPRESA_TRANSACCION 
----------------------------------------
TRUNCAR LA TABLA TRANSACCION_ENRIQUECIDA
----------------------------------------

TRUNCATE TABLE UNIVERSAL.TRANSACCION_ENRIQUECIDA; <----------- Truncamos la tabla para asegurarnos de que
                                                                    este vacia.

--------------------------------------------------------------------------------------
CREACION Y INSERCION DE DATOS PARA LA TABLA TEMPORAL TRANSACCION_PERSONA_ENRIQUECIDA_1
--------------------------------------------------------------------------------------

DROP TABLE IF EXISTS UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_1;
CREATE TEMPORARY TABLE UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_1(
ID_PERSONA STRING,
NOMBRE_PERSONA STRING,
EDAD_PERSONA INT,                                -- No es necesario COMPRIMIR una Tabla temporal
SALARIO_PERSONA DOUBLE,                          -- Como tampoco indicar el LOCATION
ID_EMPRESA_PERSONA STRING,
MONTO_TRANSACCION DOUBLE,
FECHA_TRANSACCION STRING,
ID_EMPRESA_TRANSACCION STRING
)
STORED AS PARQUET;

INSERT INTO UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_1
SELECT
  T.ID_PERSONA AS ID_PERSONA,
  P.NOMBRE AS NOMBRE_PERSONA,
  P.EDAD AS EDAD_PERSONA,
  P.SALARIO AS SALARIO_PERSONA,
  P.ID_EMPRESA AS ID_EMPRESA_PERSONA,
  T.MONTO AS MONTO_TRANSACCION,
  T.FECHA AS FECHA_TRANSACCION,
  T.ID_EMPRESA AS ID_EMPRESA_TRANSACCION
FROM
  UNIVERSAL.TRANSACCION T
JOIN 
  UNIVERSAL.PERSONA P
ON 
  T.ID_PERSONA = P.ID;

-- Desde HIVE, verificamos que se hayan insertado los datos en la tabla
SELECT * FROM UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_1 LIMIT 10;

+-----------------------------------------------+---------------------------------------------------+-------------------------------------------------+----------------------------------------------------+------------------------------------------------------+-----------------------------------------------------+-----------------------------------------------------+----------------------------------------------------------+
| transaccion_persona_enriquecida_1.id_persona  | transaccion_persona_enriquecida_1.nombre_persona  | transaccion_persona_enriquecida_1.edad_persona  | transaccion_persona_enriquecida_1.salario_persona  | transaccion_persona_enriquecida_1.id_empresa_persona | transaccion_persona_enriquecida_1.monto_transaccion | transaccion_persona_enriquecida_1.fecha_transaccion | transaccion_persona_enriquecida_1.id_empresa_transaccion |
+-----------------------------------------------+---------------------------------------------------+-------------------------------------------------+----------------------------------------------------+------------------------------------------------------+-----------------------------------------------------+-----------------------------------------------------+----------------------------------------------------------+
| 18                                            | Owen                                              | 34                                              | 4759.0                                             | 7                                                    | 1383.0                                              | 2018-01-21                                          | 3                                                        |
| 30                                            | Clayton                                           | 52                                              | 9505.0                                             | 8                                                    | 2331.0                                              | 2018-01-21                                          | 6                                                        |
| 47                                            | Vernon                                            | 35                                              | 7109.0                                             | 10                                                   | 2280.0                                              | 2018-01-21                                          | 2                                                        |
| 28                                            | Stephen                                           | 53                                              | 9469.0                                             | 8                                                    | 730.0                                               | 2018-01-21                                          | 1                                                        |
| 91                                            | Erica                                             | 32                                              | 8934.0                                             | 6                                                    | 3081.0                                              | 2018-01-21                                          | 4                                                        |
| 74                                            | Kaitlin                                           | 56                                              | 6515.0                                             | 10                                                   | 2409.0                                              | 2018-01-21                                          | 8                                                        |
| 83                                            | Giselle                                           | 45                                              | 2503.0                                             | 2                                                    | 2079.0                                              | 2018-01-21                                          | 5                                                        |
| 48                                            | Illiana                                           | 18                                              | 1454.0                                             | 8                                                    | 2543.0                                              | 2018-01-21                                          | 4                                                        |
| 15                                            | Wanda                                             | 27                                              | 1539.0                                             | 5                                                    | 1434.0                                              | 2018-01-21                                          | 6                                                        |
| 89                                            | Kelly                                             | 55                                              | 10110.0                                            | 6                                                    | 780.0                                               | 2018-01-21                                          | 4                                                        |
+-----------------------------------------------+---------------------------------------------------+-------------------------------------------------+----------------------------------------------------+------------------------------------------------------+-----------------------------------------------------+-----------------------------------------------------+----------------------------------------------------------+

-- Verificamos los tipos de datos de sus campos
DESCRIBE UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_1;

+-------------------------+------------+----------+
|        col_name         | data_type  | comment  |
+-------------------------+------------+----------+
| id_persona              | string     |          |
| nombre_persona          | string     |          |
| edad_persona            | int        |          |
| salario_persona         | double     |          |
| id_empresa_persona      | string     |          |
| monto_transaccion       | double     |          |
| fecha_transaccion       | string     |          |
| id_empresa_transaccion  | string     |          |
+-------------------------+------------+----------+


--------------------------------------------------------------------------------------
CREACION Y INSERCION DE DATOS PARA LA TABLA TEMPORAL TRANSACCION_PERSONA_ENRIQUECIDA_2
--------------------------------------------------------------------------------------

DROP TABLE IF EXISTS UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_2;
CREATE TEMPORARY TABLE UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_2(
ID_PERSONA STRING,
NOMBRE_PERSONA STRING,
EDAD_PERSONA INT,
SALARIO_PERSONA DOUBLE,
TRABAJO_PERSONA STRING,
MONTO_TRANSACCION DOUBLE,
FECHA_TRANSACCION STRING,
ID_EMPRESA_TRANSACCION STRING
)
STORED AS PARQUET;

INSERT INTO UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_2
SELECT
  T.ID_PERSONA AS ID_PERSONA,
  T.NOMBRE_PERSONA AS NOMBRE_PERSONA,
  T.EDAD_PERSONA AS EDAD_PERSONA,
  T.SALARIO_PERSONA AS SALARIO_PERSONA,
  E.NOMBRE AS TRABAJO_PERSONA,
  T.MONTO_TRANSACCION AS MONTO_TRANSACCION,
  T.FECHA_TRANSACCION AS FECHA_TRANSACCION,
  T.ID_EMPRESA_TRANSACCION AS ID_EMPRESA_TRANSACCION
FROM
  UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_1 T
JOIN 
  UNIVERSAL.EMPRESA E
ON 
  T.ID_EMPRESA_PERSONA = E.ID;

-- Desde HIVE, verificamos que se hayan insertado los datos en la tabla
SELECT * FROM UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_2 LIMIT 10;

+-----------------------------------------------+---------------------------------------------------+-------------------------------------------------+----------------------------------------------------+----------------------------------------------------+-----------------------------------------------------+-----------------------------------------------------+----------------------------------------------------------+
| transaccion_persona_enriquecida_2.id_persona  | transaccion_persona_enriquecida_2.nombre_persona  | transaccion_persona_enriquecida_2.edad_persona  | transaccion_persona_enriquecida_2.salario_persona  | transaccion_persona_enriquecida_2.trabajo_persona  | transaccion_persona_enriquecida_2.monto_transaccion | transaccion_persona_enriquecida_2.fecha_transaccion | transaccion_persona_enriquecida_2.id_empresa_transaccion |
+-----------------------------------------------+---------------------------------------------------+-------------------------------------------------+----------------------------------------------------+----------------------------------------------------+-----------------------------------------------------+-----------------------------------------------------+----------------------------------------------------------+
| 18                                            | Owen                                              | 34                                              | 4759.0                                             | Samsung                                            | 1383.0                                              | 2018-01-21                                          | 3                                                        |
| 30                                            | Clayton                                           | 52                                              | 9505.0                                             | HP                                                 | 2331.0                                              | 2018-01-21                                          | 6                                                        |
| 47                                            | Vernon                                            | 35                                              | 7109.0                                             | Sony                                               | 2280.0                                              | 2018-01-21                                          | 2                                                        |
| 28                                            | Stephen                                           | 53                                              | 9469.0                                             | HP                                                 | 730.0                                               | 2018-01-21                                          | 1                                                        |
| 91                                            | Erica                                             | 32                                              | 8934.0                                             | Google                                             | 3081.0                                              | 2018-01-21                                          | 4                                                        |
| 74                                            | Kaitlin                                           | 56                                              | 6515.0                                             | Sony                                               | 2409.0                                              | 2018-01-21                                          | 8                                                        |
| 83                                            | Giselle                                           | 45                                              | 2503.0                                             | Microsoft                                          | 2079.0                                              | 2018-01-21                                          | 5                                                        |
| 48                                            | Illiana                                           | 18                                              | 1454.0                                             | HP                                                 | 2543.0                                              | 2018-01-21                                          | 4                                                        |
| 15                                            | Wanda                                             | 27                                              | 1539.0                                             | Amazon                                             | 1434.0                                              | 2018-01-21                                          | 6                                                        |
| 89                                            | Kelly                                             | 55                                              | 10110.0                                            | Google                                             | 780.0                                               | 2018-01-21                                          | 4                                                        |
+-----------------------------------------------+---------------------------------------------------+-------------------------------------------------+----------------------------------------------------+----------------------------------------------------+-----------------------------------------------------+-----------------------------------------------------+----------------------------------------------------------+

-- Verificamos los tipos de datos de sus campos
DESCRIBE UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_2;

+-------------------------+------------+----------+
|        col_name         | data_type  | comment  |
+-------------------------+------------+----------+
| id_persona              | string     |          |
| nombre_persona          | string     |          |
| edad_persona            | int        |          |
| salario_persona         | double     |          |
| trabajo_persona         | string     |          | <---- Columna añadida 
| monto_transaccion       | double     |          |
| fecha_transaccion       | string     |          |
| id_empresa_transaccion  | string     |          |
+-------------------------+------------+----------+


--------------------------------------------------------------------------------------------
CREACION Y INSERCION DE DATOS PARA LA TABLA TEMPORAL TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA
--------------------------------------------------------------------------------------------

DROP TABLE IF EXISTS UNIVERSAL.TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA;
CREATE TEMPORARY TABLE UNIVERSAL.TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA(
ID_PERSONA STRING,
NOMBRE_PERSONA STRING,
EDAD_PERSONA INT,
SALARIO_PERSONA DOUBLE,
TRABAJO_PERSONA STRING,
MONTO_TRANSACCION DOUBLE,
FECHA_TRANSACCION STRING,
EMPRESA_TRANSACCION STRING
)
STORED AS PARQUET;

INSERT INTO UNIVERSAL.TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA
SELECT
  T.ID_PERSONA AS ID_PERSONA,
  T.NOMBRE_PERSONA AS NOMBRE_PERSONA,
  T.EDAD_PERSONA AS EDAD_PERSONA,
  T.SALARIO_PERSONA AS SALARIO_PERSONA,
  T.TRABAJO_PERSONA AS TRABAJO_PERSONA,
  T.MONTO_TRANSACCION AS MONTO_TRANSACCION,
  T.FECHA_TRANSACCION AS FECHA_TRANSACCION,
  E.NOMBRE AS EMPRESA_TRANSACCION
FROM
  UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_2 T
JOIN 
  UNIVERSAL.EMPRESA E
ON 
  T.ID_EMPRESA_TRANSACCION = E.ID;

-- Desde HIVE, verificamos que se hayan insertado los datos en la tabla
SELECT * FROM UNIVERSAL.TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA LIMIT 10;

+----------------------------------------------------+--------------------------------------------------------+------------------------------------------------------+---------------------------------------------------------+---------------------------------------------------------+-----------------------------------------------------------+-----------------------------------------------------------+-------------------------------------------------------------+
| transaccion_persona_empresa_enriquecida.id_persona | transaccion_persona_empresa_enriquecida.nombre_persona | transaccion_persona_empresa_enriquecida.edad_persona | transaccion_persona_empresa_enriquecida.salario_persona | transaccion_persona_empresa_enriquecida.trabajo_persona | transaccion_persona_empresa_enriquecida.monto_transaccion | transaccion_persona_empresa_enriquecida.fecha_transaccion | transaccion_persona_empresa_enriquecida.empresa_transaccion |
+----------------------------------------------------+--------------------------------------------------------+------------------------------------------------------+---------------------------------------------------------+---------------------------------------------------------+-----------------------------------------------------------+-----------------------------------------------------------+-------------------------------------------------------------+
| 18                                                 | Owen                                                   | 34                                                   | 4759.0                                                  | Samsung                                                 | 1383.0                                                    | 2018-01-21                                                | Apple                                                       |
| 30                                                 | Clayton                                                | 52                                                   | 9505.0                                                  | HP                                                      | 2331.0                                                    | 2018-01-21                                                | Google                                                      |
| 47                                                 | Vernon                                                 | 35                                                   | 7109.0                                                  | Sony                                                    | 2280.0                                                    | 2018-01-21                                                | Microsoft                                                   |
| 28                                                 | Stephen                                                | 53                                                   | 9469.0                                                  | HP                                                      | 730.0                                                     | 2018-01-21                                                | Walmart                                                     | 
| 91                                                 | Erica                                                  | 32                                                   | 8934.0                                                  | Google                                                  | 3081.0                                                    | 2018-01-21                                                | Toyota                                                      |
| 74                                                 | Kaitlin                                                | 56                                                   | 6515.0                                                  | Sony                                                    | 2409.0                                                    | 2018-01-21                                                | HP                                                          |
| 83                                                 | Giselle                                                | 45                                                   | 2503.0                                                  | Microsoft                                               | 2079.0                                                    | 2018-01-21                                                | Amazon                                                      |
| 48                                                 | Illiana                                                | 18                                                   | 1454.0                                                  | HP                                                      | 2543.0                                                    | 2018-01-21                                                | Toyota                                                      |
| 15                                                 | Wanda                                                  | 27                                                   | 1539.0                                                  | Amazon                                                  | 1434.0                                                    | 2018-01-21                                                | Google                                                      |
| 89                                                 | Kelly                                                  | 55                                                   | 10110.0                                                 | Google                                                  | 780.0                                                     | 2018-01-21                                                | Toyota                                                      |
+----------------------------------------------------+--------------------------------------------------------+------------------------------------------------------+---------------------------------------------------------+---------------------------------------------------------+-----------------------------------------------------------+-----------------------------------------------------------+-------------------------------------------------------------+

-- Verificamos los tipos de datos de sus campos
DESCRIBE UNIVERSAL.TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA;

+----------------------+------------+----------+
|       col_name       | data_type  | comment  |
+----------------------+------------+----------+
| id_persona           | string     |          |
| nombre_persona       | string     |          |
| edad_persona         | int        |          |
| salario_persona      | double     |          |
| trabajo_persona      | string     |          |
| monto_transaccion    | double     |          |
| fecha_transaccion    | string     |          |
| empresa_transaccion  | string     |          | <---- columna añadida 
+----------------------+------------+----------+


-----------------------------------------------------------
INSERCION FINAL DE DATOS A LA TABLA TRANSACCION_ENRIQUECIDA
-----------------------------------------------------------

ACTIVAR PARTICIONAMIENTO DINAMICO
---------------------------------
SET hive.exec.dynamic.partition=true;
SET hive.exec.dynamic.partition.mode=nonstrict;

CONFIGURAR NUMERO DE PARTICIONES =======================> Por defecto se generan 100 particiones. Pero en el caso de que
--------------------------------                          estemos cargando data historica y se generen mas particiones
SET hive.exec.max.dynamic.partitions=9999;                devolvera un error. Es por eso, que se escribe este comando
SET hive.exec.max.dynamic.partitions.pernode=9999;        dandole un numero xxx de particiones.

INSERT OVERWRITE TABLE UNIVERSAL.TRANSACCION_ENRIQUECIDA
PARTITION (FECHA_TRANSACCION)
SELECT
  ID_PERSONA,
  NOMBRE_PERSONA,
  EDAD_PERSONA,
  SALARIO_PERSONA,
  TRABAJO_PERSONA,
  MONTO_TRANSACCION,
  EMPRESA_TRANSACCION,
  FECHA_TRANSACCION
FROM
  UNIVERSAL.TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA;

-- Desde HIVE, verificamos que se hayan insertado los datos en la tabla
SELECT * FROM UNIVERSAL.TRANSACCION_ENRIQUECIDA LIMIT 10;

+-------------------------------------+-----------------------------------------+---------------------------------------+------------------------------------------+------------------------------------------+--------------------------------------------+----------------------------------------------+--------------------------------------------+
| transaccion_enriquecida.id_persona  | transaccion_enriquecida.nombre_persona  | transaccion_enriquecida.edad_persona  | transaccion_enriquecida.salario_persona  | transaccion_enriquecida.trabajo_persona  | transaccion_enriquecida.monto_transaccion  | transaccion_enriquecida.empresa_transaccion  | transaccion_enriquecida.fecha_transaccion  |
+-------------------------------------+-----------------------------------------+---------------------------------------+------------------------------------------+------------------------------------------+--------------------------------------------+----------------------------------------------+--------------------------------------------+
| 18                                  | Owen                                    | 34                                    | 4759.0                                   | Samsung                                  | 1383.0                                     | Apple                                        | 2018-01-21                                 |
| 30                                  | Clayton                                 | 52                                    | 9505.0                                   | HP                                       | 2331.0                                     | Google                                       | 2018-01-21                                 |
| 47                                  | Vernon                                  | 35                                    | 7109.0                                   | Sony                                     | 2280.0                                     | Microsoft                                    | 2018-01-21                                 |
| 28                                  | Stephen                                 | 53                                    | 9469.0                                   | HP                                       | 730.0                                      | Walmart                                      | 2018-01-21                                 |
| 91                                  | Erica                                   | 32                                    | 8934.0                                   | Google                                   | 3081.0                                     | Toyota                                       | 2018-01-21                                 |
| 74                                  | Kaitlin                                 | 56                                    | 6515.0                                   | Sony                                     | 2409.0                                     | HP                                           | 2018-01-21                                 |
| 83                                  | Giselle                                 | 45                                    | 2503.0                                   | Microsoft                                | 2079.0                                     | Amazon                                       | 2018-01-21                                 |
| 48                                  | Illiana                                 | 18                                    | 1454.0                                   | HP                                       | 2543.0                                     | Toyota                                       | 2018-01-21                                 |
| 15                                  | Wanda                                   | 27                                    | 1539.0                                   | Amazon                                   | 1434.0                                     | Google                                       | 2018-01-21                                 |
| 89                                  | Kelly                                   | 55                                    | 10110.0                                  | Google                                   | 780.0                                      | Toyota                                       | 2018-01-21                                 |
+-------------------------------------+-----------------------------------------+---------------------------------------+------------------------------------------+------------------------------------------+--------------------------------------------+----------------------------------------------+--------------------------------------------+

-- Verificamos los tipos de datos de sus campos
DESCRIBE UNIVERSAL.TRANSACCION_ENRIQUECIDA;

+--------------------------+------------+----------+
|         col_name         | data_type  | comment  |
+--------------------------+------------+----------+
| id_persona               | int        |          |
| nombre_persona           | string     |          |
| edad_persona             | int        |          |
| salario_persona          | double     |          |
| trabajo_persona          | string     |          |
| monto_transaccion        | double     |          |
| empresa_transaccion      | string     |          |
| fecha_transaccion        | string     |          |
|                          | NULL       | NULL     |
| # Partition Information  | NULL       | NULL     |
| # col_name               | data_type  | comment  |
| fecha_transaccion        | string     |          |
+--------------------------+------------+----------+


--------------------------------
ELIMINACION DE TABLAS TEMPORALES
--------------------------------

DROP TABLE IF EXISTS UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_1;
DROP TABLE IF EXISTS UNIVERSAL.TRANSACCION_PERSONA_ENRIQUECIDA_2;
DROP TABLE IF EXISTS UNIVERSAL.TRANSACCION_PERSONA_EMPRESA_ENRIQUECIDA;

#### Paso 7 - Crear la capa SMART

In [ ]:
TRANSACCION                                 TRANSACCION_ENRIQUECIDA                                                            TRANSACCION_POR_EDAD
 ________                                              ________    ID_PERSONA                                 ___________            ________     EDAD_PERSONA     
|__|__|__| ID_PERSONA                                 |__|__|__|   NOMBRE_PERSONA                            |           |          |__|__|__|    CANTIDAD_TRANSACCIONES
|__|__|__| ID_EMPRESA ----.                .--------> |__|__|__|   EDAD_PERSONA                    .-------> | PROCESO 2 | -------> |__|__|__|    SUMA_MONTO_TRANSACCIONES
|__|__|__| MONTO          |                |          |__|__|__|   SALARIO_PERSONA                 |         |___________|          |__|__|__|
           FECHA          |                |                       TRABAJO_PERSONA  ---------------|
PERSONA                   |                |                       MONTO_TRANSACCION               | 
 ________  ID             |           _____|_____                  FECHA_TRANSACCION               |                           TRANSACCION_POR_TRABAJO
|__|__|__| NOMBRE         |          |           |                 EMPRESA_TRANSACCION             |          ___________            ________     TRABAJO_PERSONA
|__|__|__| TELEFONO ------|--------> | PROCESO 1 |                                                 |         |           |          |__|__|__|    CANTIDAD_TRANSACCIONES
|__|__|__| CORREO         |          |___________|                                                 |-------> | PROCESO 3 | -------> |__|__|__|    SUMA_MONTO_TRANSACCIONES
           FECHA_INGRESO  |                                                                        |         |___________|          |__|__|__|
           EDAD           |                                                                        | 
           SALARIO        |                                                                        | 
           ID_EMPRESA     |                                                                        |                                                                           
EMPRESA                   |                                                                        |                           TRANSACCION_POR_EMPRESA      
 ________                 |                                                                        |          ___________            ________    EMPRESA_TRANSACCION
|__|__|__| ID             |                                                                        |         |           |          |__|__|__|   CANTIDAD_TRANSACCIONES
|__|__|__| NOMBRE --------'                                                                        '-------> | PROCESO 4 | -------> |__|__|__|   SUMA_MONTO_TRANSACCIONES
|__|__|__|                                                                                                   |___________|          |__|__|__|


----------------------------------------------------------------------------------------------------------------

DESAROLLO
=========
                                              ------------------------
                                                TRANSACCION POR EDAD    
                                              ------------------------

-----------
CREAR TABLA
-----------

DROP TABLE IF EXISTS SMART.TRANSACCION_POR_EDAD;
CREATE TABLE SMART.TRANSACCION_POR_EDAD(
EDAD_PERSONA INT,
CANTIDAD_TRANSACCIONES INT,
SUMA_MONTO_TRANSACCIONES DOUBLE
)
STORED AS PARQUET
LOCATION '/user/proyectos/proyecto3/database/smart/transaccion_por_edad'
TBLPROPERTIES ("parquet.compression"="SNAPPY");

-------------------------------
INSERCION DE DATOS EN LA TABLA
-------------------------------

INSERT OVERWRITE TABLE SMART.TRANSACCION_POR_EDAD
SELECT
  EDAD_PERSONA,
  COUNT(MONTO_TRANSACCION),
  SUM(MONTO_TRANSACCION)
FROM
  UNIVERSAL.TRANSACCION_ENRIQUECIDA
GROUP BY
  EDAD_PERSONA;

-- Desde HIVE, verificamos que se hayan insertado los datos en la tabla
SELECT * FROM SMART.TRANSACCION_POR_EDAD LIMIT 10;

+------------------------------------+----------------------------------------------+------------------------------------------------+
| transaccion_por_edad.edad_persona  | transaccion_por_edad.cantidad_transacciones  | transaccion_por_edad.suma_monto_transacciones  |
+------------------------------------+----------------------------------------------+------------------------------------------------+
| 18                                 | 4707                                         | 1.0478251E7                                    |
| 19                                 | 9513                                         | 2.1448462E7                                    |
| 22                                 | 11677                                        | 2.613166E7                                     |
| 23                                 | 4670                                         | 1.0415767E7                                    |
| 24                                 | 9379                                         | 2.1185021E7                                    |
| 25                                 | 4621                                         | 1.0358377E7                                    |
| 26                                 | 6891                                         | 1.5612084E7                                    |
| 27                                 | 11766                                        | 2.642914E7                                     |
| 28                                 | 2397                                         | 5419577.0                                      |
| 29                                 | 4607                                         | 1.0278431E7                                    |
+------------------------------------+----------------------------------------------+------------------------------------------------+

-- Verificamos los tipos de datos de sus campos
DESCRIBE SMART.TRANSACCION_POR_EDAD;

+---------------------------+------------+----------+
|         col_name          | data_type  | comment  |
+---------------------------+------------+----------+
| edad_persona              | int        |          |
| cantidad_transacciones    | int        |          |
| suma_monto_transacciones  | double     |          |
+---------------------------+------------+----------+


                                              --------------------------
                                                TRANSACCION POR TRABAJO    
                                              --------------------------

-----------
CREAR TABLA
-----------

DROP TABLE IF EXISTS SMART.TRANSACCION_POR_TRABAJO;
CREATE TABLE SMART.TRANSACCION_POR_TRABAJO(
TRABAJO_PERSONA STRING,
CANTIDAD_TRANSACCIONES INT,
SUMA_MONTO_TRANSACCIONES DOUBLE
)
STORED AS PARQUET
LOCATION '/user/proyectos/proyecto3/database/smart/transaccion_por_trabajo'
TBLPROPERTIES ("parquet.compression"="SNAPPY");

-------------------------------
INSERCION DE DATOS EN LA TABLA
-------------------------------

INSERT OVERWRITE TABLE SMART.TRANSACCION_POR_TRABAJO
SELECT
  TRABAJO_PERSONA,
  COUNT(MONTO_TRANSACCION),
  SUM(MONTO_TRANSACCION)
FROM
  UNIVERSAL.TRANSACCION_ENRIQUECIDA
GROUP BY
  TRABAJO_PERSONA;

-- Desde HIVE, verificamos que se hayan insertado los datos en la tabla
SELECT * FROM SMART.TRANSACCION_POR_TRABAJO LIMIT 10;

+------------------------------------------+-------------------------------------------------+---------------------------------------------------+
| transaccion_por_trabajo.trabajo_persona  | transaccion_por_trabajo.cantidad_transacciones  | transaccion_por_trabajo.suma_monto_transacciones  |
+------------------------------------------+-------------------------------------------------+---------------------------------------------------+
| Amazon                                   | 32911                                           | 7.3819971E7                                       |
| Apple                                    | 25923                                           | 5.801854E7                                        |
| Google                                   | 30583                                           | 6.8591761E7                                       |
| HP                                       | 21247                                           | 4.7466547E7                                       |
| IBM                                      | 14124                                           | 3.1569729E7                                       |
| Microsoft                                | 33098                                           | 7.4393407E7                                       |
| Samsung                                  | 21160                                           | 4.7443803E7                                       |
| Sony                                     | 20917                                           | 4.7099878E7                                       |
| Toyota                                   | 18979                                           | 4.2356472E7                                       |
| Walmart                                  | 16098                                           | 3.6413955E7                                       |
+------------------------------------------+-------------------------------------------------+---------------------------------------------------+

-- Verificamos los tipos de datos de sus campos
DESCRIBE SMART.TRANSACCION_POR_TRABAJO;

+---------------------------+------------+----------+
|         col_name          | data_type  | comment  |
+---------------------------+------------+----------+
| trabajo_persona           | string     |          |
| cantidad_transacciones    | int        |          |
| suma_monto_transacciones  | double     |          |
+---------------------------+------------+----------+


                                              ---------------------------
                                                TRANSACCION POR EMPRESA    
                                              ---------------------------

-----------
CREAR TABLA
-----------

DROP TABLE IF EXISTS SMART.TRANSACCION_POR_EMPRESA;
CREATE TABLE SMART.TRANSACCION_POR_EMPRESA(
EMPRESA_TRANSACCION STRING,
CANTIDAD_TRANSACCIONES INT,
SUMA_MONTO_TRANSACCIONES DOUBLE
)
STORED AS PARQUET
LOCATION '/user/proyectos/proyecto3/database/smart/transaccion_por_empresa'
TBLPROPERTIES ("parquet.compression"="SNAPPY");

-------------------------------
INSERCION DE DATOS EN LA TABLA
-------------------------------

INSERT OVERWRITE TABLE SMART.TRANSACCION_POR_EMPRESA
SELECT
  EMPRESA_TRANSACCION,
  COUNT(MONTO_TRANSACCION),
  SUM(MONTO_TRANSACCION)
FROM
  UNIVERSAL.TRANSACCION_ENRIQUECIDA
GROUP BY
  EMPRESA_TRANSACCION;

-- Desde HIVE, verificamos que se hayan insertado los datos en la tabla
SELECT * FROM SMART.TRANSACCION_POR_EMPRESA LIMIT 10;

+----------------------------------------------+-------------------------------------------------+---------------------------------------------------+
| transaccion_por_empresa.empresa_transaccion  | transaccion_por_empresa.cantidad_transacciones  | transaccion_por_empresa.suma_monto_transacciones  |
+----------------------------------------------+-------------------------------------------------+---------------------------------------------------+
| Amazon                                       | 23443                                           | 5.2402645E7                                       |
| Apple                                        | 23575                                           | 5.342402E7                                        |
| Google                                       | 23406                                           | 5.2402979E7                                       |
| HP                                           | 23382                                           | 5.2488235E7                                       |
| IBM                                          | 23581                                           | 5.2589229E7                                       |
| Microsoft                                    | 23472                                           | 5.2680068E7                                       |
| Samsung                                      | 23522                                           | 5.2882909E7                                       |
| Sony                                         | 23616                                           | 5.296073E7                                        |
| Toyota                                       | 23389                                           | 5.2121975E7                                       |
| Walmart                                      | 23654                                           | 5.3221273E7                                       |
+----------------------------------------------+-------------------------------------------------+---------------------------------------------------+

-- Verificamos los tipos de datos de sus campos
DESCRIBE SMART.TRANSACCION_POR_EMPRESA;

+---------------------------+------------+----------+
|         col_name          | data_type  | comment  |
+---------------------------+------------+----------+
| empresa_transaccion       | string     |          |
| cantidad_transacciones    | int        |          |
| suma_monto_transacciones  | double     |          |
+---------------------------+------------+----------+


                                      --------------------------------------------
                                      EJECUCION DE SCRIPTS DE SOLUCION POR CONSOLA
                                      --------------------------------------------

PROCESO 1:
         ________________________________________________________________________________________________
        |                                                                                                |         
        |   beeline -u jdbc:hive2:// -f /home/main/proceso_1.sql --hiveconf "PARAM_USERNAME=XXXXX"       |
        |________________________________________________________________________________________________|

PROCESO 2:
         ________________________________________________________________________________________________
        |                                                                                                |         
        |   beeline -u jdbc:hive2:// -f /home/main/proceso_2.sql --hiveconf "PARAM_USERNAME=XXXXX"       |
        |________________________________________________________________________________________________|

PROCESO 3:
         ________________________________________________________________________________________________
        |                                                                                                |         
        |   beeline -u jdbc:hive2:// -f /home/main/proceso_3.sql --hiveconf "PARAM_USERNAME=XXXXX"       |
        |________________________________________________________________________________________________|    

PROCESO 4:
         ________________________________________________________________________________________________
        |                                                                                                |         
        |   beeline -u jdbc:hive2:// -f /home/main/proceso_4.sql --hiveconf "PARAM_USERNAME=XXXXX"       |
        |________________________________________________________________________________________________| 

Si en el caso de que existiese MÁS DE 1 PARAMETRO, se agrega a continuacion del primero:
         __________________________________________________________________________________________________________________
        |                                                                                                                  |         
        |   beeline -u jdbc:hive2:// -f /home/main/proceso.sql --hiveconf "PARAM_USERNAME=XXXXX" --hiveconf "PARAM2=XXXX"  |
        |__________________________________________________________________________________________________________________|         
        
          